# Country names for the hotel bookings dataset

**Data sources:** cleaned bookings `data/processed/hotel_bookings_clean.csv` (Part 1, task 1.6), from Antonio, de Almeida and Nunes (2019); country names from the Python package **pycountry 26.2.16**, which packages the ISO 3166 country lists of the Debian *iso-codes* project (LGPL-2.1).
**Date:** 2026-11-05
**Authors:** Group 3, Presentation and Visualization (Infovis2627)
**Description:** The dataset identifies the guests' countries only with ISO three-letter codes (PRT, GBR…). The persona of Part 2 reads countries by name, so this notebook (Part 3, task 3.2) builds a lookup table from code to English name and saves it in `data/processed/country_names.csv`. It does not change the bookings file.

---

## Contents

1. [Data loading](#1.-Data-loading)
2. [Initial exploration: which codes need a name](#2.-Initial-exploration:-which-codes-need-a-name)
3. [Cleaning and preparation: building the lookup](#3.-Cleaning-and-preparation:-building-the-lookup)
4. [Analysis: checks of the lookup](#4.-Analysis:-checks-of-the-lookup)
5. [Accessible visualisation](#5.-Accessible-visualisation)
6. [Export](#6.-Export)
7. [Conclusions](#Conclusions)

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import pandas as pd
import pycountry

CLEAN_FILE  = Path('../../data/processed/hotel_bookings_clean.csv')
LOOKUP_FILE = Path('../../data/processed/country_names.csv')
OUTPUT_DIR  = Path('country_names_output')
OUTPUT_DIR.mkdir(exist_ok=True)

PYCOUNTRY_VERSION = '26.2.16'   # version installed and used for this table
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| pycountry', PYCOUNTRY_VERSION)

Python 3.13.18 | pandas 3.0.5 | pycountry 26.2.16


## 1. Data loading

In [2]:
# ── Load the cleaned bookings (country, plus two fields for the checks of section 4) ─
bookings = pd.read_csv(CLEAN_FILE, usecols=['country', 'is_canceled', 'deposit_type'], keep_default_na=False)
print(f'Rows: {len(bookings):,}')
bookings['country'].value_counts().head()

Rows: 119,201


country
PRT    48476
GBR    12118
FRA    10401
ESP     8560
DEU     7285
Name: count, dtype: int64

## 2. Initial exploration: which codes need a name

In [3]:
# ── Codes in the data, with their number of bookings ─────────────────────────
codes = bookings['country'].value_counts().rename_axis('country').rename('bookings').reset_index()

# Is each code in the current ISO 3166-1 list?
codes['in current ISO list'] = codes['country'].map(lambda code: pycountry.countries.get(alpha_3=code) is not None)
print('Distinct codes:', len(codes))
codes[~codes['in current ISO list']]

Distinct codes: 177


,country,bookings,in current ISO list
20,Unknown,478,False
128,TMP,3,False


**Finding:** 175 of the 177 values are current ISO 3166-1 codes. The other two are `TMP`, the code that East Timor used until 2002 (now `TLS`), and `Unknown`, the label that cleaning rule 5 gave to the missing countries.

## 3. Cleaning and preparation: building the lookup

In [4]:
# ── Name of each code ─────────────────────────────────────────────────────────

def country_name(code):
    """English name of a three-letter code, and where the name comes from."""
    country = pycountry.countries.get(alpha_3=code)
    if country is not None:
        # The short common name when ISO has one (e.g. 'Iran' instead of 'Iran, Islamic Republic of')
        return getattr(country, 'common_name', country.name), 'ISO 3166-1 (current)'
    for old_country in pycountry.historic_countries:
        if getattr(old_country, 'alpha_3', None) == code:
            return old_country.name, 'ISO 3166-3 (former code)'
    if code == 'Unknown':
        return 'Unknown', 'Cleaning rule 5 (missing country)'
    return code, 'No name found'

names = codes['country'].map(country_name)
codes['country_name'] = names.str[0]
codes['name_source'] = names.str[1]
codes[['country', 'country_name', 'name_source', 'bookings']].head(12)

,country,country_name,name_source,bookings
0,PRT,Portugal,ISO 3166-1 (current),48476
1,GBR,United Kingdom,ISO 3166-1 (current),12118
2,FRA,France,ISO 3166-1 (current),10401
3,ESP,Spain,ISO 3166-1 (current),8560
4,DEU,Germany,ISO 3166-1 (current),7285
5,ITA,Italy,ISO 3166-1 (current),3761
6,IRL,Ireland,ISO 3166-1 (current),3374
7,BEL,Belgium,ISO 3166-1 (current),2342
8,CHN,China,ISO 3166-1 (current),2278
9,BRA,Brazil,ISO 3166-1 (current),2222


### Decision: which name to show

**Context:** ISO gives an official name (for example "Korea, Republic of") and, for some countries, a shorter common name ("South Korea").

**Decision taken:** the common name is used when it exists, otherwise the official name, because the persona reads them quickly and the course's cognitive accessibility guidance asks for plain language. Nothing is typed by hand: every name comes from the ISO tables in pycountry.

**Alternatives considered:** UN regions were not added (the students chose names only), so small countries will still be grouped as "Other" in the dashboard.

> ⚠️ **Limitation:** `CN` was already recoded as `CHN` (China) by cleaning rule 5; that is an assumption of Part 1, not of this table.

## 4. Analysis: checks of the lookup

In [5]:
# ── Every code has exactly one name, and every booking can be joined ─────────
print('Codes without a name:', (codes['name_source'] == 'No name found').sum())
print('Duplicated codes:', codes['country'].duplicated().sum())
joined = bookings.merge(codes[['country', 'country_name']], on='country', how='left')
print('Bookings without a name after the join:', joined['country_name'].isna().sum(), 'of', f'{len(joined):,}')
codes['name_source'].value_counts()

Codes without a name: 0
Duplicated codes: 0
Bookings without a name after the join: 0 of 119,201


name_source
ISO 3166-1 (current)                 175
Cleaning rule 5 (missing country)      1
ISO 3166-3 (former code)               1
Name: count, dtype: int64

In [6]:
# ── Effect on the Part 1 results: top 10 countries after cleaning, with names ─
top_10 = codes.head(10)[['country', 'country_name', 'bookings']].copy()
top_10['% of bookings'] = (100 * top_10['bookings'] / len(bookings)).round(1)
top_10

,country,country_name,bookings,% of bookings
0,PRT,Portugal,48476,40.7
1,GBR,United Kingdom,12118,10.2
2,FRA,France,10401,8.7
3,ESP,Spain,8560,7.2
4,DEU,Germany,7285,6.1
5,ITA,Italy,3761,3.2
6,IRL,Ireland,3374,2.8
7,BEL,Belgium,2342,2.0
8,CHN,China,2278,1.9
9,BRA,Brazil,2222,1.9


In [7]:
# ── Portugal's cancellation rate, with and without Non Refund deposits ───────
# (cleaning rule 1 keeps the duplicated rows, so the dashboard needs a note about this effect)
portugal = bookings[bookings['country'] == 'PRT']
print(f"Overall cancellation rate: {100 * bookings['is_canceled'].mean():.1f} %")
print(f"Portugal: {100 * portugal['is_canceled'].mean():.1f} %")
print(f"Portugal without Non Refund deposits: {100 * portugal.loc[portugal['deposit_type'] != 'Non Refund', 'is_canceled'].mean():.1f} %")

Overall cancellation rate: 37.1 %
Portugal: 56.7 %
Portugal without Non Refund deposits: 39.0 %


**Finding:** after cleaning rule 5 joined `CN` to `CHN`, **China** enters the ten largest markets (2,278 bookings, 1.9 %), Brazil moves down to tenth place and the Netherlands leaves the top ten; the Part 1 country ranking (made on the raw data) changes slightly. Portugal's cancellation rate (56.7 %, against 37.1 % overall) falls to 39.0 % without its non-refundable bookings, which are mostly repeated rows, so the dashboard must say so next to any cancellation rate by country (the students' decision for rule 1).

## 5. Accessible visualisation

No chart is needed: the result is a lookup table. Its effect is visible later, when the dashboard labels countries by name instead of by code.

## 6. Export

In [8]:
# ── Export ────────────────────────────────────────────────────────────────────
lookup = codes[['country', 'country_name', 'name_source']].sort_values('country')
lookup.to_csv(LOOKUP_FILE, index=False)
codes.to_csv(OUTPUT_DIR / 'country_codes_with_bookings.csv', index=False)

# Read it back to check that it loads correctly
check = pd.read_csv(LOOKUP_FILE, keep_default_na=False)
print(f'{LOOKUP_FILE.name}: {len(check)} rows, columns {list(check.columns)}')
check.head()

country_names.csv: 177 rows, columns ['country', 'country_name', 'name_source']


,country,country_name,name_source
0,ABW,Aruba,ISO 3166-1 (current)
1,AGO,Angola,ISO 3166-1 (current)
2,AIA,Anguilla,ISO 3166-1 (current)
3,ALB,Albania,ISO 3166-1 (current)
4,AND,Andorra,ISO 3166-1 (current)


## Conclusions

### Main findings
1. All 177 country values now have an English name: 175 from the current ISO 3166-1 list, 1 former code (`TMP`, East Timor) and the `Unknown` label.
2. Every one of the 119,201 cleaned bookings can be joined to a name.

### Limitations
- Names are in English, as the dashboard; they follow ISO's common names, which may differ from other usual spellings.

### Suggested next steps
- Use `country_names.csv` in Part 6 to label countries; group the small ones as "Other".